# ECMWF ENS — Download

**Model:** ECMWF Ensemble Forecast System  
**Type:** Physics-based probabilistic — 51 members (1 control + 50 perturbed)  
**Cycles:** 00Z and 12Z only  
**Range:** 0–360h at 6-hourly

## Files produced

```
data/YYYY-MM-DD_HHZ/
    ens_prob_precip_d1-d5.grib2   P(precip > 1mm) and P(precip > 5mm), 24h windows
    ens_prob_wave_d1-d5.grib2     P(SWH > 2m / 3m / 4m), 12-hourly steps
    ens_tc_tracks.bufr            ENS TC tracks (when active)
```

In [ ]:
from ecmwf.opendata import Client
from pathlib import Path

BASE_DIR = Path("data")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print("Ready. BASE_DIR:", BASE_DIR.resolve())

In [ ]:
client = Client(source="ecmwf", model="ifs")
latest_run  = client.latest()
latest_time = latest_run.hour

run_label  = f"{latest_run.strftime('%Y-%m-%d')}_{latest_time:02d}Z"
output_dir = BASE_DIR / run_label
output_dir.mkdir(parents=True, exist_ok=True)

print("Latest run  :", latest_run)
print("Cycle (UTC) :", latest_time)
print("Output dir  :", output_dir.resolve())

In [ ]:
# ── ENS precipitation probability — 24h windows, D1–D5 ───────────────────────
client.retrieve(
    time=latest_time, stream="enfo", type="ep",
    step=["0-24", "12-36", "24-48", "36-60", "48-72",
          "60-84", "72-96", "84-108", "96-120"],
    param=["tpg1", "tpg5"],
    target=str(output_dir / "ens_prob_precip_d1-d5.grib2"),
)
print("Saved: ens_prob_precip_d1-d5.grib2")

In [ ]:
# ── ENS wave height probability — 12-hourly, 0–120h ─────────────────────────
client.retrieve(
    time=latest_time, stream="waef", type="ep",
    step=list(range(12, 121, 12)),
    param=["swhg2", "swhg3", "swhg4"],
    target=str(output_dir / "ens_prob_wave_d1-d5.grib2"),
)
print("Saved: ens_prob_wave_d1-d5.grib2")

In [ ]:
# ── ENS TC tracks — when active ──────────────────────────────────────────────
try:
    client.retrieve(time=latest_time, stream="enfo", type="tf", step=240,
        target=str(output_dir / "ens_tc_tracks.bufr"))
    print("Saved: ens_tc_tracks.bufr")
except ValueError:
    print("No active TCs this run — file not available")

In [ ]:
# ── NEW: ENS control forecast surface — 6-hourly, 0–120h ─────────────────────
steps_6h = list(range(0, 121, 6))

ENS_SFC = {}
ENS_SFC["thermo"]      = ["2t", "2d", "skt", "sstk"]
ENS_SFC["wind"]        = ["10u", "10v", "10fg", "100u", "100v"]
ENS_SFC["pressure"]    = ["msl", "sp"]
ENS_SFC["moisture"]    = ["tp", "cp", "tcc", "lcc", "mcc", "hcc", "tcwv", "ro"]
ENS_SFC["energy"]      = ["ssrd", "strd", "ttr", "blh"]
ENS_SFC["instability"] = ["mucape", "mucin"]

client.retrieve(
    time=latest_time, step=steps_6h, type="cf", stream="enfo",
    param=[p for grp in ENS_SFC.values() for p in grp],
    target=str(output_dir / "ens_cf_sfc_f000-f120_6h.grib2"),
)
print("Saved: ens_cf_sfc_f000-f120_6h.grib2")
print(f"  Groups: { {k: len(v) for k, v in ENS_SFC.items()} }")

In [ ]:
# ── NEW: ENS control forecast upper-air — 12-hourly, 0–120h ──────────────────
ENS_PL = {}
ENS_PL["thermo"]   = ["t", "gh"]
ENS_PL["wind"]     = ["u", "v", "w"]
ENS_PL["moisture"] = ["q", "r"]
ENS_PL["dynamics"] = ["vo", "d"]

client.retrieve(
    time=latest_time, step=list(range(0, 121, 12)), type="cf", stream="enfo",
    param=[p for grp in ENS_PL.values() for p in grp],
    levtype="pl", levelist=[1000, 925, 850, 700, 600, 500, 400, 300, 250, 200],
    target=str(output_dir / "ens_cf_pl_f000-f120_12h.grib2"),
)
print("Saved: ens_cf_pl_f000-f120_12h.grib2")

In [ ]:
# ── NEW: ENS ensemble mean surface — 6-hourly, 0–120h ────────────────────────
ENS_MEAN_SFC = {}
ENS_MEAN_SFC["thermo"]   = ["2t"]
ENS_MEAN_SFC["wind"]     = ["10u", "10v"]
ENS_MEAN_SFC["pressure"] = ["msl"]
ENS_MEAN_SFC["moisture"] = ["tp"]

client.retrieve(
    time=latest_time, step=steps_6h, type="em", stream="enfo",
    param=[p for grp in ENS_MEAN_SFC.values() for p in grp],
    target=str(output_dir / "ens_mean_sfc_f000-f120_6h.grib2"),
)
print("Saved: ens_mean_sfc_f000-f120_6h.grib2")